# Phần bổ sung – Notebook gọi công cụ và xử lý ngoại lệ
Ngày tạo: 23/10/2026. Chạy lần lượt 4 ô Python 3 bên dưới.
Không cần API key, tài khoản IBM hay thư viện ngoài Python chuẩn.
Bộ chọn hành động là luật; đây không phải LLM, không phải chạy lại notebook gốc IBM.
Dữ liệu thời tiết và giá vé là giả định. Các kiểm thử chỉ đánh giá bộ điều phối và công cụ.
Giữ riêng notebook này để không thay đổi notebook 8 ô của báo cáo cũ.

In [1]:
import json, math

# Dữ liệu giả định, không phải thời tiết hay giá thực tế.
WEATHER = {"Athens": "rain", "Crete": "sun"}
VENUES = [("Bảo tàng", "indoor", 12),
          ("Công viên", "outdoor", 0)]

def get_weather(city):
    if not isinstance(city, str):
        raise TypeError("city phải là chuỗi")
    return {"city": city, "weather": WEATHER[city]}

def find_places(kind, budget):
    if type(budget) not in (int, float):
        raise TypeError("budget phải là số")
    if not math.isfinite(budget) or budget < 0:
        raise ValueError("budget phải hữu hạn và không âm")
    if kind not in ("indoor", "outdoor"):
        raise ValueError("kind không được hỗ trợ")
    return [name for name, group, price in VENUES
            if group == kind and price <= budget]

TOOLS = {"get_weather": get_weather,
         "find_places": find_places}

def dispatch(call):
    if call["tool"] not in TOOLS:
        raise PermissionError("Công cụ không được phép")
    if not isinstance(call["args"], dict):
        raise TypeError("args phải là dictionary")
    return TOOLS[call["tool"]](**call["args"])

print("Công cụ:", ", ".join(TOOLS))

Công cụ: get_weather, find_places


In [2]:
def choose_action(goal, memory):
    # Bộ quyết định theo luật; không phải LLM.
    if "weather" not in memory:
        return {"tool": "get_weather", "save_as": "weather",
                "args": {"city": goal["city"]}}
    if "places" not in memory:
        kind = ("indoor" if memory["weather"]["weather"] == "rain"
                else "outdoor")
        return {"tool": "find_places", "save_as": "places",
                "args": {"kind": kind, "budget": goal["budget"]}}
    return {"finish": True, "goal_met": bool(memory["places"]),
            "answer": memory["places"] or ["Không có phương án phù hợp"]}


def run_task(goal, planner=choose_action, max_steps=5):
    memory, events = {}, []
    for step in range(1, max_steps + 1):
        call = planner(goal, memory)
        if call.get("finish"):
            return {"status": "done" if call["goal_met"] else "no_match",
                    "answer": call["answer"], "events": events}
        event = {"step": step, "tool": call["tool"], "args": call["args"]}
        try:
            value = dispatch(call)
        except (KeyError, TypeError, ValueError, PermissionError) as error:
            event["error"] = type(error).__name__
            events.append(event)
            return {"status": "tool_error", "events": events}
        memory[call["save_as"]] = value
        event["result"] = value
        events.append(event)
    return {"status": "step_limit", "events": events}

print("Đã tạo bộ quyết định và bộ điều phối.")

Đã tạo bộ quyết định và bộ điều phối.


In [3]:
tests = []
def record(name, passed):
    assert passed, name
    tests.append(name)
    print("ĐẠT:", name)

rain = run_task({"city": "Athens", "budget": 15})
sun = run_task({"city": "Crete", "budget": 0})
record("Mưa: chọn bảo tàng", rain["answer"] == ["Bảo tàng"])
record("Nắng: chọn công viên", sun["answer"] == ["Công viên"])
record("Thiếu ngân sách", run_task({"city": "Athens", "budget": 5})["status"] == "no_match")
for name, goal, expected in [
    ("Ngân sách âm", {"city": "Athens", "budget": -1}, "ValueError"),
    ("Thành phố lạ", {"city": "Unknown", "budget": 15}, "KeyError"),
    ("Ngân sách là chuỗi", {"city": "Athens", "budget": "15"}, "TypeError"),
    ("Ngân sách NaN", {"city": "Athens", "budget": float("nan")}, "ValueError")]:
    result = run_task(goal)
    record(name, result["status"] == "tool_error" and
           result["events"][-1]["error"] == expected)

bad_tool = lambda g, m: {"tool": "delete_files", "args": {}, "save_as": "x"}
bad_args = lambda g, m: {"tool": "get_weather", "args": {"wrong": "Athens"}, "save_as": "x"}
repeat = lambda g, m: {"tool": "get_weather", "args": {"city": "Athens"}, "save_as": "x"}
record("Chặn công cụ lạ", run_task({}, bad_tool)["events"][0]["error"] == "PermissionError")
record("Chặn sai tham số", run_task({}, bad_args)["events"][0]["error"] == "TypeError")
limited = run_task({}, repeat, max_steps=2)
record("Dừng quyết định lặp", limited["status"] == "step_limit" and len(limited["events"]) == 2)
print(f"KẾT QUẢ: {len(tests)}/10 kiểm thử đạt.")

ĐẠT: Mưa: chọn bảo tàng
ĐẠT: Nắng: chọn công viên
ĐẠT: Thiếu ngân sách
ĐẠT: Ngân sách âm
ĐẠT: Thành phố lạ
ĐẠT: Ngân sách là chuỗi
ĐẠT: Ngân sách NaN
ĐẠT: Chặn công cụ lạ
ĐẠT: Chặn sai tham số
ĐẠT: Dừng quyết định lặp
KẾT QUẢ: 10/10 kiểm thử đạt.


In [4]:
print("VẾT CHẠY: ATHENS, NGÂN SÁCH 15")
for event in rain["events"]:
    print(json.dumps(event, ensure_ascii=False))
print("Trạng thái:", rain["status"])
print("Kết quả:", rain["answer"])
print("THIẾU NGÂN SÁCH:", run_task({"city": "Athens", "budget": 5})["status"])
print("VÒNG LẶP:", limited["status"], "-", len(limited["events"]), "lời gọi")

VẾT CHẠY: ATHENS, NGÂN SÁCH 15
{"step": 1, "tool": "get_weather", "args": {"city": "Athens"}, "result": {"city": "Athens", "weather": "rain"}}
{"step": 2, "tool": "find_places", "args": {"kind": "indoor", "budget": 15}, "result": ["Bảo tàng"]}
Trạng thái: done
Kết quả: ['Bảo tàng']
THIẾU NGÂN SÁCH: no_match
VÒNG LẶP: step_limit - 2 lời gọi
